# Week 3 - Assignment: Voice Agent Development

From now on, we start to hands on buiding Research Voice Agent, truly useful AI Research Assistants must listen, understand, and respond with voice. **we will give you some simple introduction code as a starter, feel free to write your own code or do optimization.**

## 📚 Learning Objectives this week
to build a simple Voice Agent, we need these following knowledge.

* **1. Speech Recognition (ASR):** Convert audio to text using models like Whisper or Google Speech-to-Text.
* **2. Dialogue Generation with LLMs:** Feed transcribed user input into LLM (e.g. LLaMA 3) and generate natural language responses.
* **3. Text-to-Speech (TTS):** Use a TTS engine (CozyVoice) to convert generated responses into spoken audio.
* **4. FastAPI for API Serving:** Create a web server with FastAPI to handle audio file uploads and return voice responses.
* **5. Conversation State Management:** Track conversation history to enable multi-turn interaction.
* **6. Low-Latency Real-Time Processing:** Use asynchronous functions to reduce inference time and improve response experience.

---


> ✅ You do NOT need Docker. Just ensure your local Python environment works.

---

## 🧪 Project: Build an Local Voice Assistant

### 🎯 Goal:

Develop a real-time voice chatbot that can:

1. Take audio input via HTTP,
2. Transcribe audio to text (ASR),
3. Generate a response using LLM,
4. Convert the response back to speech (TTS),
5. Support 5-turn conversational memory.

---

### Step 1: FastAPI Skeleton

Create a simple FastAPI server that accepts an audio file via POST and returns an audio file in response:


here is the official guidance of FastAPI [fastapi](https://fastapi.tiangolo.com/)

In [ ]:
import os
import shutil
from urllib.parse import quote
from dotenv import load_dotenv

load_dotenv()

from fastapi import FastAPI, UploadFile, File, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.staticfiles import StaticFiles

from asr import transcribe_audio
from llm import generate_response, get_history, clear_history
from tts import synthesize_speech

app = FastAPI(title="Voice Agent", description="5-turn voice chatbot: ASR → LLM → TTS")

app.mount("/static", StaticFiles(directory="static"), name="static")


def _cleanup(path: str):
    parent = os.path.dirname(path)
    if os.path.isdir(parent):
        shutil.rmtree(parent, ignore_errors=True)


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
async def index():
    with open("static/index.html") as f:
        return f.read()


@app.post("/chat/", summary="Send audio, get audio reply")
async def chat_endpoint(
    file: UploadFile = File(..., description="Audio file (wav/mp3/webm)"),
    background_tasks: BackgroundTasks = None,
):
    audio_bytes = await file.read()
    if not audio_bytes:
        raise HTTPException(status_code=400, detail="Empty audio file")

    user_text = transcribe_audio(audio_bytes)
    print(f"[User]      {user_text}")

    bot_text = generate_response(user_text)
    print(f"[Assistant] {bot_text}")

    wav_path = synthesize_speech(bot_text)
    if background_tasks:
        background_tasks.add_task(_cleanup, wav_path)

    return FileResponse(
        wav_path,
        media_type="audio/wav",
        headers={
            "X-User-Text": quote(user_text),
            "X-Bot-Text":  quote(bot_text),
            "Access-Control-Expose-Headers": "X-User-Text, X-Bot-Text",
        },
    )


@app.get("/history/", summary="Get conversation history")
async def history_endpoint():
    return {"turns": len(get_history()) // 2, "history": get_history()}


@app.delete("/history/", summary="Clear conversation history")
async def clear_history_endpoint():
    clear_history()
    return {"message": "Conversation history cleared"}


@app.get("/health/")
async def health():
    return {"status": "ok"}

Run your server:

```bash
uvicorn main:app --reload
```

Test it with `curl`, Postman, or a custom frontend.

### Step 2: ASR (Speech Recognition)

Use OpenAI Whisper to transcribe the uploaded audio to text:

In [ ]:
import whisper
import os
import tempfile

_model = None


def _get_model():
    global _model
    if _model is None:
        print("[ASR] Loading Whisper medium model...")
        _model = whisper.load_model("medium")  # uses cached ~/.cache/whisper/medium.pt
        print("[ASR] Model loaded.")
    return _model


def transcribe_audio(audio_bytes: bytes) -> str:
    model = _get_model()
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        tmp_path = f.name
    try:
        result = model.transcribe(tmp_path)
        return result["text"].strip()
    finally:
        os.remove(tmp_path)


# --- Pipeline test ---
import os
os.chdir("voice_agent")

with open("../test_data/audio/sample-1.mp3", "rb") as f:
    audio_bytes = f.read()

user_text = transcribe_audio(audio_bytes)
print(f"[ASR Output] {user_text}")

**Output:**
```
[ASR] Loading Whisper medium model...
[ASR] Model loaded.
[ASR Output] So we pay so that they didn't, you know, that you some people said to son,
             could end up in so instead of caught in a musical, or just not told a virtual,
             and just end up turning child, that's the...
```

Whisper `medium` model is cached locally at `~/.cache/whisper/medium.pt`. No download needed.

Add it to the `/chat/` route:

In [ ]:
user_text = transcribe_audio(audio_bytes)

Print `user_text` for debugging.


### Step 3: Response Generation (LLM)

Generate context-aware responses using Llama 3. Use HuggingFace `pipeline` to call LLaMA 3 or similar models:


In [ ]:
from transformers import pipeline

llm = pipeline("text-generation", model="meta-llama/Llama-3-8B")

conversation_history = []

def generate_response(user_text):
    conversation_history.append({"role": "user", "text": user_text})
    # Construct prompt from history
    prompt = ""
    for turn in conversation_history[-5:]:
        prompt += f"{turn['role']}: {turn['text']}\n"
    outputs = llm(prompt, max_new_tokens=100)
    bot_response = outputs[0]["generated_text"]
    conversation_history.append({"role": "assistant", "text": bot_response})
    return bot_response


Call in route:

In [ ]:
bot_text = generate_response(user_text)


---



### Step 4: TTS (Text to Speech)


In [ ]:
import subprocess
import tempfile
import os


def synthesize_speech(text: str) -> str:
    """Convert text to speech using macOS built-in `say` command.
    Returns path to a temporary WAV file (caller responsible for cleanup).
    """
    tmp_dir = tempfile.mkdtemp()
    aiff_path = os.path.join(tmp_dir, "response.aiff")
    wav_path  = os.path.join(tmp_dir, "response.wav")

    # macOS built-in TTS → AIFF
    subprocess.run(["say", "-o", aiff_path, "--", text], check=True, capture_output=True)

    # Convert AIFF → WAV with ffmpeg
    subprocess.run(["ffmpeg", "-y", "-i", aiff_path, wav_path], check=True, capture_output=True)

    os.remove(aiff_path)
    return wav_path


# --- Test ---
wav = synthesize_speech("Hello, this is a test of the text to speech system.")
print(f"[TTS] Output: {wav}")
print(f"[TTS] File size: {os.path.getsize(wav):,} bytes")

**Output:**
```
[TTS] Output: /var/folders/.../response.wav
[TTS] File size: 213,418 bytes
```

Uses macOS built-in `say` command — no external API or model download needed. Output is converted from AIFF to WAV via `ffmpeg`.

In [ ]:
import sys
sys.path.append('third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import CosyVoice, CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, load_vllm=False, fp16=False)

# NOTE if you want to reproduce the results on https://funaudiollm.github.io/cosyvoice2, please add text_frontend=False during inference
# zero_shot usage
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


### Use it in the route:

In [ ]:
pythonaudio_path = synthesize_speech(bot_text)



---

---

## ✅ Deliverables

* [x] A runnable FastAPI project with `/chat/` endpoint
* [x] A working voice assistant that handles **5-turn** multi-round conversations
* [x] Code with clear structure and modular components (ASR, LLM, TTS)
* [x] A screen recording demo of real-time 5-turn interaction: [`HW/recording.mov`](HW/recording.mov)

---

## Implementation Summary

| Component | Tool | Notes |
|-----------|------|-------|
| **ASR** | OpenAI Whisper `medium` | Local model, cached at `~/.cache/whisper/medium.pt` |
| **LLM** | Google Gemini 2.5 Flash | Via `google-genai` SDK, 5-turn memory window |
| **TTS** | macOS `say` + ffmpeg | Built-in, no API key required |
| **Server** | FastAPI + Uvicorn | `/chat/` POST endpoint, `/history/` GET/DELETE |
| **Frontend** | Vanilla HTML/CSS/JS | Hold-to-record mic UI, conversation display |

### Project Structure
```
voice_agent/
├── main.py          # FastAPI server + routing
├── asr.py           # Whisper transcription module
├── llm.py           # Gemini LLM + 5-turn conversation memory
├── tts.py           # macOS say TTS module
├── requirements.txt # Python dependencies
├── .env.example     # GOOGLE_API_KEY template
└── static/
    └── index.html   # Browser UI with hold-to-record mic

HW/
├── 1.png            # Demo screenshot
├── 2.png            # Demo screenshot
└── recording.mov    # Screen recording demo
```

### Run
```bash
cd voice_agent
uvicorn main:app --reload
# Open http://127.0.0.1:8000
```

### Demo Evidence

**5-turn conversation screenshots (Turn 3/5 and Turn 5/5):**

![demo-1](HW/1.png)
![demo-2](HW/2.png)

**Screen recording:** [`HW/recording.mov`](HW/recording.mov)

<video src="HW/recording.mov" controls width="720"></video>


In [ ]:

@app.post("/chat/")
async def chat_endpoint(file: UploadFile = File(...)):
    audio_bytes = await file.read()
    user_text = transcribe_audio(audio_bytes)
    bot_text = generate_response(user_text)
    audio_path = synthesize_speech(bot_text)
    return FileResponse(audio_path, media_type="audio/wav")


---

## ✅ Final Submission Checklist

* [x] A runnable FastAPI project with `/chat/` endpoint
* [x] A working voice assistant that handles **5-turn** multi-round conversations
* [x] Code with clear structure and modular components (ASR, LLM, TTS)
* [x] A screen recording demo: `HW/recording.mov`
* [x] Conversation memory display and prompt formatting logic are implemented in the frontend/backend.

---
